In [2]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Sequence
import operator

# Core LangGraph components, including checkpoints
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver

# LLM and messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

# Load API keys and set up tracing
load_dotenv()
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Intro to LangGraph"

# -------------------------------
# Define the State
# -------------------------------
class GraphState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]


# -------------------------------
# Define Nodes
# -------------------------------
def call_llm(state: GraphState):
    print("--- Calling LLM ---")
    llm = ChatOpenAI(model="gpt-4o")
    response = llm.invoke(state['messages'])
    # Add the AI's response to the message list
    return {"messages": [response]}


def human_approval(state: GraphState):
    print("\n--- Awaiting Human Approval ---")
    last_message = state['messages'][-1]
    print(f"AI Response to Review: {last_message.content}")
    # The graph pauses AFTER this node
    return {}


# -------------------------------
# Build the Graph
# -------------------------------
workflow = StateGraph(GraphState)
workflow.add_node("llm", call_llm)
workflow.add_node("approve", human_approval)

workflow.add_edge(START, "llm")
workflow.add_edge("llm", "approve")
workflow.add_edge("approve", END)


# -------------------------------
# Use context manager for SqliteSaver
# -------------------------------
with SqliteSaver.from_conn_string(":memory:") as memory:
    # Compile the graph WITH interrupts
    app = workflow.compile(
        checkpointer=memory,
        interrupt_after=["approve"]
    )

    # --- Run the Graph with Human Feedback ---
    config = {"configurable": {"thread_id": "feedback-thread-1"}}

    print("--- Starting graph execution ---")
    # Initial invocation - runs 'llm' and 'approve', then pauses
    initial_state = app.invoke(
        {"messages": [HumanMessage(content="Write a short poem about LangGraph.")]},
        config
    )

    print("\n--- Graph Paused for Feedback ---")
    print("Current state (at breakpoint):")
    for msg in initial_state['messages']:
        print(f"- {msg.type}: {msg.content}")

    # --- Simulating Human Feedback ---
    print("\n--- Adding Human Feedback ---")
    current_values = app.get_state(config)
    current_values.values['messages'] += [
        HumanMessage(content="Correction: Make the poem rhyme.")
    ]
    app.update_state(config, current_values.values)

    print("\n--- State Updated with Human Feedback ---")
    print("New state:")
    for msg in current_values.values['messages']:
        print(f"- {msg.type}: {msg.content}")

    # --- Resume the Graph ---
    print("\n--- Resuming graph execution ---")
    final_state = app.invoke(None, config)

    print("\n--- Graph Finished ---")
    print("Final state:")
    for msg in final_state['messages']:
        print(f"- {msg.type}: {msg.content}")


--- Starting graph execution ---
--- Calling LLM ---

--- Awaiting Human Approval ---
AI Response to Review: In the realm of code where ideas weave,  
LangGraph stands, where queries believe.  
A tapestry spun with nodes and lines,  
Charting knowledge in intricate designs.  

Whispers of logic in every thread,  
Connections where curiosity is fed.  
A chorus of data in structured grace,  
LangGraph guides like a mentor's face.  

From abstract thought to expression clear,  
Language and knowledge both revere.  
In LangGraph's arms, insights unfold,  
A narrative of data, bold and untold.  

With every vertex, a story blooms,  
Illuminating minds, dissolving gloom.  
In the dance of queries, answers sing,  
LangGraph's legacy—a future to bring.

--- Graph Paused for Feedback ---
Current state (at breakpoint):
- human: Write a short poem about LangGraph.
- ai: In the realm of code where ideas weave,  
LangGraph stands, where queries believe.  
A tapestry spun with nodes and lines,  
Cha